# Gemini Document Prompt

Upload a document to Gemini, apply a prompt, and print the extracted text.

In [16]:
from pathlib import Path
import os

from dotenv import find_dotenv, load_dotenv

# Fill these values before running the next cells.
DOCUMENT = Path.home() / "Desktop/CNAS.jpeg"
PROMPT = "Extrage fieldurile din document, si pentru fiecare spune si cat esti de sigur ca asta scrie acolo."
MODEL = "gemini-3.1-flash-lite-preview"
KEEP_FILE = False
WAIT_SECONDS = 60

# Jupyter kernels often don't inherit env vars from your terminal session.
# Prefer a local .env file (see .env.example) or a globally-set env var.
load_dotenv(find_dotenv(usecwd=True))
API_KEY = os.getenv("GEMINI_API_KEY")


In [17]:
import time

try:
    from google import genai
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Missing dependency: pip install google-genai") from exc


def wait_until_active(client: genai.Client, file_name: str, max_wait_seconds: int) -> None:
    start = time.time()
    while True:
        current = client.files.get(name=file_name)
        state = getattr(current, "state", None)
        state_value = getattr(state, "name", str(state)) if state is not None else "UNKNOWN"

        if state_value == "ACTIVE":
            return
        if state_value == "FAILED":
            raise RuntimeError(f"File processing failed for {file_name}.")
        if time.time() - start > max_wait_seconds:
            raise TimeoutError(
                f"Timed out waiting for uploaded file to become ACTIVE (>{max_wait_seconds}s)."
            )
        time.sleep(2)


In [18]:
if not API_KEY:
    raise ValueError("Set GEMINI_API_KEY or GOOGLE_API_KEY before running.")
if not DOCUMENT.exists():
    raise FileNotFoundError(f"Document does not exist: {DOCUMENT}")
if not DOCUMENT.is_file():
    raise ValueError(f"Document is not a file: {DOCUMENT}")

client = genai.Client(api_key=API_KEY)
uploaded = client.files.upload(file=DOCUMENT)
wait_until_active(client, uploaded.name, max_wait_seconds=WAIT_SECONDS)

try:
    response = client.models.generate_content(
        model=MODEL,
        contents=[uploaded, PROMPT],
    )
    output_text = getattr(response, "text", None)
    if not output_text:
        raise RuntimeError("Gemini returned no text output.")
    print(output_text)
finally:
    if not KEEP_FILE:
        client.files.delete(name=uploaded.name)

Iată extragerea datelor din certificatul medical furnizat. Menționez că am tratat cu prudență valorile scrise de mână, care pot fi ambigue.

| Câmp / Informație | Valoare extrasă | Nivel de încredere |
| :--- | :--- | :--- |
| **Seria și Numărul** | CCMAT 1384563 | Foarte ridicat |
| **Luna/Anul** | Iulie 2025 | Ridicat |
| **Cod indemnizație** | 06 (Urgență) | Ridicat |
| **Cod Numeric Personal (CNP)** | 2990730013569 | Ridicat |
| **Număr Înregistrare (RC/FO)** | 30591 | Ridicat |
| **Data acordării** | 31.07.25 | Ridicat |
| **Număr zile** | 02 | Ridicat |
| **Perioada (De la - Până la)** | 30.07.25 - 31.07.25 | Ridicat |
| **Cod diagnostic** | 689 | Ridicat |
| **Tip diagnostic** | Acut (bifat) | Ridicat |
| **Cod Convenție unitate** | 432 | Mediu |
| **CAS Emitentă** | Alba (ABA) | Mediu |
| **CUI Unitate Sanitară** | 27719257 | Ridicat |
| **Data primirii** | 31.07.2025 | Ridicat |

---

### Observații importante:
1. **Confidențialitate:** Ai lăsat vizibile nume și adrese în imag

In [ ]:
#GEMINI 2.5 FLASH result

if not API_KEY:
    raise ValueError("Set GEMINI_API_KEY or GOOGLE_API_KEY before running.")
if not DOCUMENT.exists():
    raise FileNotFoundError(f"Document does not exist: {DOCUMENT}")
if not DOCUMENT.is_file():
    raise ValueError(f"Document is not a file: {DOCUMENT}")

client = genai.Client(api_key=API_KEY)
uploaded = client.files.upload(file=DOCUMENT)
wait_until_active(client, uploaded.name, max_wait_seconds=WAIT_SECONDS)

try:
    response = client.models.generate_content(
        model=MODEL,
        contents=[uploaded, PROMPT],
    )
    output_text = getattr(response, "text", None)
    if not output_text:
        raise RuntimeError("Gemini returned no text output.")
    print(output_text)
finally:
    if not KEEP_FILE:
        client.files.delete(name=uploaded.name)


Here are the extracted fields from the document:

*   **Urgenţă medico-chirurgicală (checkbox status and value)**: Checked, 35
*   **În continuare (checkbox status)**: Checked
*   **Data acordării (day, month, year)**: 31/07/2023
*   **Nr. zile**: 02
*   **De la zi/lună/an (day, month, year)**: 30/07/2023
*   **Până la zi/lună/an (day, month, year)**: 31/07/2023
*   **Cod diagnostic**: 649
*   **ACUT (checkbox status)**: Checked
*   **CRONIC (checkbox status)**: Unchecked
*   **Valabil pentru luna (month and year)**: Iulie 2023
*   **Seria CCMAT Nr.**: 1384563
*   **Cod indemnizație (1-17)**: 060002
*   **Asigurat în evidenţă la CAS**: MBR
*   **Cod numeric personal**: 219904230073565
*   **Nr. înreg. (RC/FO)**: 30659
*   **Ambulator/Internat în spital**: Empty
*   **Secţia**: Empty
*   **Concediu medical la externare**: Empty
*   **Unitatea sanitară emitentă**: Dr. Juju Marcela
*   **Nr. convenţie**: 1932
*   **cu CAS**: MBR
*   **Cod electronic de înregistrare**: 217119251
*   **CAS 